# 02 · Discrete 1D Image Derivatives and Boundary Visualizations

In computer vision, a digital grayscale image is represented as a 2D discrete function:

$$I: \{0, 1, \dots, H-1\} \times \{0, 1, \dots, W-1\} \to [0, 255]$$

where each spatial coordinate $(i, j)$ represents a row and column index, and $I[i, j]$ is the pixel intensity. 

Because digital images are sampled on a discrete lattice rather than a continuous continuum, derivatives must be approximated using **finite differences**:
- Single-sided forward difference: $\Delta x = 1$
- Symmetric central difference: $\Delta x = 2$

In this notebook, we explore:
1. Treating image rows and columns as 1D discrete functions.
2. Computing horizontal ($x$-direction) and vertical ($y$-direction) derivatives.
3. Solving the visualization problem for negative gradient values using **magnitude** ($|d|$) and **midpoint level-shifting** ($d + 128$).
4. Implementing pure Python algorithms and comparing them with high-performance vectorized NumPy operations.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Locate image fixture in ../data/nature.jpg
data_path = Path("../data/nature.jpg")
assert data_path.is_file(), f"Image not found at {data_path.resolve()}"

# Read image in grayscale mode (values in [0, 255])
img = cv2.imread(str(data_path), cv2.IMREAD_GRAYSCALE)
height, width = img.shape
print(f"Loaded image '{data_path.name}': Height = {height}, Width = {width}, dtype = {img.dtype}")

# Convert to nested Python list for pure Python algorithmic walkthroughs
image_list = img.tolist()
assert len(image_list) == height and len(image_list[0]) == width

## 1. 1D Single-Sided Forward Difference (Row-wise and Column-wise)

For each row $i$, the horizontal forward difference with step size $\Delta x = 1$ is:

$$\Delta_x I[i, j] = I[i, j+1] - I[i, j] \quad \text{for } 0 \le j < W-1$$

Similarly, for each column $j$, the vertical forward difference with $\Delta y = 1$ is:

$$\Delta_y I[i, j] = I[i+1, j] - I[i, j] \quad \text{for } 0 \le i < H-1$$

### The Visualization Problem
Because pixel values are non-negative ($0 \le I \le 255$), the difference $d = I[i, j+1] - I[i, j]$ spans $[-255, +255]$:
- A transition from dark ($0$) to bright ($255$) yields $+255$.
- A transition from bright ($255$) to dark ($0$) yields $-255$.
- A uniform flat region yields $0$.

If saved directly into an 8-bit unsigned integer buffer (`uint8`), negative numbers underflow ($0 - 50 = 206$).

Two canonical schemes resolve this:
1. **Magnitude (Absolute Value):** $d_{\text{mag}} = |d|$. All sharp transitions appear bright, regardless of direction.
2. **Midpoint Level-Shifting:** $d_{\text{shift}} = \text{clip}(d + 127.5, 0, 255)$. Flat regions map to neutral gray ($128$), positive slopes are bright ($>128$), and negative slopes are dark ($<128$).


In [ ]:
def derivative_x_abs(data: list[list[float]], height: int, width: int) -> list[list[float]]:
    """Pure Python horizontal derivative using absolute magnitude."""
    result = [[0.0] * width for _ in range(height)]
    for i in range(height):
        for j in range(width - 1):
            d_val = abs(data[i][j + 1] - data[i][j])
            result[i][j] = float(min(d_val, 255.0))
    return result


def derivative_x_shifted(data: list[list[float]], height: int, width: int) -> list[list[float]]:
    """Pure Python horizontal derivative using midpoint level-shifting (d + 127.5)."""
    result = [[128.0] * width for _ in range(height)]
    for i in range(height):
        for j in range(width - 1):
            d_val = data[i][j + 1] - data[i][j]
            shifted = d_val + 127.5
            result[i][j] = float(max(0.0, min(255.0, shifted)))
    return result


def derivative_y_shifted(data: list[list[float]], height: int, width: int) -> list[list[float]]:
    """Pure Python vertical derivative using midpoint level-shifting."""
    result = [[128.0] * width for _ in range(height)]
    for j in range(width):
        for i in range(height - 1):
            d_val = data[i + 1][j] - data[i][j]
            shifted = d_val + 127.5
            result[i][j] = float(max(0.0, min(255.0, shifted)))
    return result

In [ ]:
# Test on a small 50x50 crop for pure Python execution speed
crop = [row[:50] for row in image_list[:50]]
crop_dx_abs = derivative_x_abs(crop, 50, 50)
crop_dx_shift = derivative_x_shifted(crop, 50, 50)

print(
    f"Shifted crop range: min = {min(map(min, crop_dx_shift)):.1f}, max = {max(map(max, crop_dx_shift)):.1f}"
)
assert min(map(min, crop_dx_shift)) >= 0.0 and max(map(max, crop_dx_shift)) <= 255.0

## 2. 1D Symmetric Central Difference

Using a step size $\Delta x = 2$ centered around pixel $j$:

$$\delta_x I[i, j] = \frac{I[i, j+1] - I[i, j-1]}{2} \quad \text{for } 1 \le j < W-1$$

Central difference cancels the linear first-order error term in the Taylor expansion:
$$I(x + h) = I(x) + h I'(x) + \frac{h^2}{2}I''(x) + \mathcal{O}(h^3)$$
$$I(x - h) = I(x) - h I'(x) + \frac{h^2}{2}I''(x) - \mathcal{O}(h^3)$$
$$I(x + h) - I(x - h) = 2h I'(x) + \mathcal{O}(h^3) \implies I'(x) = \frac{I(x + h) - I(x - h)}{2h} + \mathcal{O}(h^2)$$


In [ ]:
def derivative_x_central(data: list[list[float]], height: int, width: int) -> list[list[float]]:
    """Pure Python horizontal central difference with level-shifting."""
    result = [[128.0] * width for _ in range(height)]
    for i in range(height):
        for j in range(1, width - 1):
            d_val = (data[i][j + 1] - data[i][j - 1]) / 2.0
            shifted = d_val + 127.5
            result[i][j] = float(max(0.0, min(255.0, shifted)))
    return result


def derivative_y_central(data: list[list[float]], height: int, width: int) -> list[list[float]]:
    """Pure Python vertical central difference with level-shifting."""
    result = [[128.0] * width for _ in range(height)]
    for j in range(width):
        for i in range(1, height - 1):
            d_val = (data[i + 1][j] - data[i - 1][j]) / 2.0
            shifted = d_val + 127.5
            result[i][j] = float(max(0.0, min(255.0, shifted)))
    return result

## 3. High-Performance Vectorized NumPy Implementation

While nested Python loops are didactic, production vision systems vectorize discrete differences using NumPy array slicing.


In [ ]:
# NumPy Vectorized Implementations
def np_forward_dx(image: np.ndarray, mode="shifted") -> np.ndarray:
    diff = np.zeros_like(image, dtype=np.float64)
    diff[:, :-1] = image[:, 1:].astype(np.float64) - image[:, :-1].astype(np.float64)
    if mode == "abs":
        return np.clip(np.abs(diff), 0, 255).astype(np.uint8)
    return np.clip(diff + 127.5, 0, 255).astype(np.uint8)


def np_central_dx(image: np.ndarray) -> np.ndarray:
    diff = np.zeros_like(image, dtype=np.float64)
    diff[:, 1:-1] = (image[:, 2:].astype(np.float64) - image[:, :-2].astype(np.float64)) / 2.0
    return np.clip(diff + 127.5, 0, 255).astype(np.uint8)


def np_central_dy(image: np.ndarray) -> np.ndarray:
    diff = np.zeros_like(image, dtype=np.float64)
    diff[1:-1, :] = (image[2:, :].astype(np.float64) - image[:-2, :].astype(np.float64)) / 2.0
    return np.clip(diff + 127.5, 0, 255).astype(np.uint8)


dx_abs = np_forward_dx(img, mode="abs")
dx_shift = np_forward_dx(img, mode="shifted")
dx_central = np_central_dx(img)
dy_central = np_central_dy(img)

# Verify alignment between pure Python and NumPy on crop
py_cnt = np.array(derivative_x_central(crop, 50, 50))[1:49, 1:49]
np_cnt = dx_central[:50, :50][1:49, 1:49].astype(np.float64)
assert np.allclose(py_cnt, np_cnt, atol=1.0)
print("NumPy and pure Python central difference implementations agree perfectly!")

In [ ]:
# Multi-panel visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].imshow(img, cmap="gray")
axes[0, 0].set_title("Original Grayscale Image", fontsize=12)
axes[0, 0].axis("off")

axes[0, 1].imshow(dx_abs, cmap="gray")
axes[0, 1].set_title("Horizontal Forward Difference |d| (Magnitude)", fontsize=12)
axes[0, 1].axis("off")

axes[1, 0].imshow(dx_central, cmap="gray")
axes[1, 0].set_title("Horizontal Central Difference (d/2 + 128)", fontsize=12)
axes[1, 0].axis("off")

axes[1, 1].imshow(dy_central, cmap="gray")
axes[1, 1].set_title("Vertical Central Difference (d/2 + 128)", fontsize=12)
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()